## DistilGpt2 Load

In [1]:
from transformers import AutoTokenizer , AutoModelForCausalLM 

# Auto Tokenizer is for the convert text to a token 
# Automodelforcasualllm is for loads a text generating model 

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

print(model)

/Users/debajyotihazra/Documents/Fine Tune/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 76/76 [00:00<00:00, 12616.45it/s]


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


GPT2LMHeadModel
│
├── transformer
│   ├── Embedding
│   ├── 6 × GPT2Block
│   │   ├── Attention
│   │   ├── LayerNorm
│   │   └── MLP
│   │
│   └── LayerNorm
│
└── lm_head

In [2]:
#text to Tokens 

text = " I am learning Fine Tune " 
tokens = tokenizer(text)
print(tokens)

{'input_ids': [314, 716, 4673, 17867, 42587, 220], 'attention_mask': [1, 1, 1, 1, 1, 1]}


In [3]:
# Output generate 

text = "I am learning"

token = tokenizer(
    text,
    return_tensors="pt"
)

output = model.generate(
    **token,
    max_new_tokens=20
)

print(tokenizer.decode(output[0]))

I am learning to be a good person. I am not a bad person. I am not a bad person.


### Dataset 

In [4]:
data = [
    {
        "question":"what is python?",
        "answer":"python is high level programing language."
    },
    {
        "question": "What is RAG?",
        "answer": "RAG retrieves relevant information and gives it to a language model."
    },
    {
        "question": "What is FastAPI?",
        "answer": "FastAPI is a Python framework for building APIs."
    }
    
]

import json 
with open(file='dataset.json' , mode='w') as file :
    json.dump(data,file,indent=4)

In [5]:
from datasets import load_dataset 
dataset  = load_dataset(
    "json",
    data_files="dataset.json"
)
print(dataset)

print(f'the first line for train : {dataset['train'][0]}')

Generating train split: 3 examples [00:00, 201.44 examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 3
    })
})
the first line for train : {'question': 'what is python?', 'answer': 'python is high level programing language.'}


In [6]:
# Format change for data

def format_data(example):
    return {
        "text":
            f"Question :{example['question']}\n"
            f'answer :{example['answer']}'
    }
dataset = dataset.map(format_data)
print(dataset['train'][0])

Map: 100%|██████████| 3/3 [00:00<00:00, 431.78 examples/s]

{'question': 'what is python?', 'answer': 'python is high level programing language.', 'text': 'Question :what is python?\nanswer :python is high level programing language.'}


In [9]:
print(tokenizer)
print(type(tokenizer))
tokenizer.pad_token = tokenizer.eos_token

GPT2Tokenizer(name_or_path='distilgpt2', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})
<class 'transformers.models.gpt2.tokenization_gpt2.GPT2Tokenizer'>


In [10]:
# Tokenizer 
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenize_dataset = dataset.map(tokenize)

print(tokenize_dataset["train"][0])

Map: 100%|██████████| 3/3 [00:00<00:00, 133.00 examples/s]

{'question': 'what is python?', 'answer': 'python is high level programing language.', 'text': 'Question :what is python?\nanswer :python is high level programing language.', 'input_ids': [24361, 1058, 10919, 318, 21015, 30, 198, 41484, 1058, 29412, 318, 1029, 1241, 1430, 278, 3303, 13, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256

### Create label 

In [14]:
def add_label(example):
    example['labels'] = [token if mask == 1 else -100 for token , mask in zip (example['input_ids'] , example['attention_mask']) ]
    return example 

tokenize_dataset = tokenize_dataset.map(add_label)

Map: 100%|██████████| 3/3 [00:00<00:00, 58.17 examples/s]


In [15]:
print(tokenize_dataset["train"][0]["labels"])

[24361, 1058, 10919, 318, 21015, 30, 198, 41484, 1058, 29412, 318, 1029, 1241, 1430, 278, 3303, 13, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]


### Trainer 

In [16]:
from transformers import TrainingArguments, Trainer

In [ ]:
training_args = TrainingArguments(
    output_dir="./fine_tune_model",
    num_train_epochs=10 , 
    per_device_train_batch_size=1 , 
    learning_rate=5e-5,
    logging_steps=1 , 
    save_strategy='no',
    report_to='none'
)

trainer = Trainer(model=model, args=training_args,train_dataset=tokenize_dataset['train'])

In [18]:
trainer.train()

/Users/debajyotihazra/Documents/Fine Tune/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,4.916479
2,3.919073
3,4.671265
4,4.494155
5,3.099763
6,3.216512
7,3.597269
8,2.358502
9,2.423501
10,2.243721


TrainOutput(global_step=30, training_loss=2.2169768810272217, metrics={'train_runtime': 13.6458, 'train_samples_per_second': 2.198, 'train_steps_per_second': 2.198, 'total_flos': 979862814720.0, 'train_loss': 2.2169768810272217, 'epoch': 10.0})

In [20]:
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
prompt = "Question :what is python?\nanswer :"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

output = model.generate(
    **inputs,
    max_new_tokens=30
)

print(tokenizer.decode(output[0]))

/Users/debajyotihazra/Documents/Fine Tune/.venv/lib/python3.14/site-packages/transformers/generation/utils.py:2770: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on mps. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('mps') before running `.generate()`.
  warnings.warn(


Question :what is python?
answer :python is a programming language. It is a programming language. It is a programming language. It is a programming language. It is a programming language.
